# Inference Test

In [13]:
import pandas as pd
import numpy as np
import joblib
import sys
import os

from datetime import datetime

# Add project root to path
sys.path.append(os.path.abspath(os.path.join('..')))

import mlflow
from src import config

print("✅ Libraries loaded. Ready for inference.")

✅ Libraries loaded. Ready for inference.


### Load & Preprocess Validation Data

In [14]:
# Load the validation interim created in 00_data_collection
df_validation = pd.read_parquet("../data/interim/water_quality_mvp_validation.parquet")

# Load the fitted preprocessor from training
preprocessor = joblib.load("../models/preprocessor.joblib")

print(f"Validation data shape: {df_validation.shape}")
print(f"Preprocessor loaded ✅")

Validation data shape: (200, 10)
Preprocessor loaded ✅


In [15]:
df_validation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Latitude                       200 non-null    float64       
 1   Longitude                      200 non-null    float64       
 2   Sample Date                    200 non-null    datetime64[ns]
 3   swir22                         181 non-null    float64       
 4   NDMI                           181 non-null    float64       
 5   MNDWI                          181 non-null    float64       
 6   pet                            200 non-null    float64       
 7   Total Alkalinity               0 non-null      float64       
 8   Electrical Conductance         0 non-null      float64       
 9   Dissolved Reactive Phosphorus  0 non-null      float64       
dtypes: datetime64[ns](1), float64(9)
memory usage: 15.8 KB


In [16]:
# Define target columns (same as training)
TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Identify features for preprocessing
df_val_features = df_validation.drop(columns=TARGET_COLS, errors='ignore').copy()

# Extract time features (same as training)
# df_val_features['Sample Date'] = pd.to_datetime(df_val_features['Sample Date'], errors='coerce', dayfirst=True)
# df_val_features['Month'] = df_val_features['Sample Date'].dt.month.astype(str)

# Drop redundant columns (same as training)
df_val_features = df_val_features.drop(columns=['Sample Date', 'Latitude', 'Longitude'], errors='ignore')

# Apply preprocessor (transform only, don't fit)
X_val_processed = preprocessor.transform(df_val_features)

# Convert to DataFrame with proper column names
X_val_df = pd.DataFrame(X_val_processed, columns=preprocessor.get_feature_names_out())

print(f"Processed validation features shape: {X_val_df.shape}")
print(f"Feature count: {len(preprocessor.get_feature_names_out())}")

Processed validation features shape: (200, 4)
Feature count: 4


### Load Best Model

In [17]:
import mlflow

# Set tracking URI (same as training)
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("WaterQuality")

# Dictionary to store best model for each target
best_models = {}

print("Loading best models from the LAST 9 RUNS in MLFlow...\n")

# Fetch ALL runs in the experiment as a Pandas DataFrame
all_runs = mlflow.search_runs(experiment_names=["WaterQuality"])

if all_runs.empty:
    print("⚠️ No runs found in experiment.")
else:
    # Sort by time (newest first) and isolate the 3x3 throwdown we just did
    all_runs = all_runs.sort_values("start_time", ascending=False)
    recent_9_runs = all_runs.head(9)

    # 3. Iterate through targets to find the best model strictly within those 9 runs
    for target in TARGET_COLS:
        # Filter our isolated 9 runs for the current target
        target_runs = recent_9_runs[recent_9_runs['params.target'] == target]
        
        if len(target_runs) == 0:
            print(f"⚠️  No recent runs found for {target}")
            continue
        
        # Find run with highest R2 score within this clean batch
        best_run_idx = target_runs['metrics.r2'].idxmax()
        best_run = target_runs.loc[best_run_idx]
        
        run_id = best_run['run_id']
        r2_score = best_run['metrics.r2']
        model_name = best_run['params.model']
        
        # Load the model
        model_uri = f"runs:/{run_id}/model"
        model = mlflow.sklearn.load_model(model_uri)
        
        best_models[target] = model
        
        print(f"✅ {target}")
        print(f"   Run ID: {run_id}")
        print(f"   Model: {model_name}")
        print(f"   R2 Score: {r2_score:.4f}\n")

print(f"🎯 Total models loaded: {len(best_models)}/3")

Loading best models from the LAST 9 RUNS in MLFlow...



✅ Total Alkalinity
   Run ID: cdb03f71626d42c5b0c002ad3229cb93
   Model: RandomForest
   R2 Score: 0.5603



✅ Electrical Conductance
   Run ID: 8e246cf8dfb14435bcda780be0c5be54
   Model: RandomForest
   R2 Score: 0.6117



✅ Dissolved Reactive Phosphorus
   Run ID: 3f1eaf25885740adb25c3d2292937f22
   Model: RandomForest
   R2 Score: 0.5546

🎯 Total models loaded: 3/3


### Make Prediction

In [18]:
# Make predictions using MLFlow models
predictions_dict = {}

for target in TARGET_COLS:
    if target in best_models:
        print(f"Predicting {target}...")
        model = best_models[target]
        y_pred = model.predict(X_val_df)
        predictions_dict[target] = y_pred
        print(f"  ✅ {len(y_pred)} predictions generated\n")
    else:
        print(f"  ⚠️ Skipping {target} - model not loaded\n")

print(f"✅ All predictions complete!")

Predicting Total Alkalinity...
  ✅ 200 predictions generated

Predicting Electrical Conductance...
  ✅ 200 predictions generated

Predicting Dissolved Reactive Phosphorus...
  ✅ 200 predictions generated

✅ All predictions complete!


### Save Submission File

In [19]:
# Load original submission template (with Lat/Lon/Date)
submission_base = pd.read_csv("../data/raw/submission_template.csv")

# Add predictions to submission
submission_final = submission_base.copy()

for target in TARGET_COLS:
    if target in predictions_dict:
        submission_final[target] = predictions_dict[target]
    else:
        submission_final[target] = np.nan

# Save submission with dynamic naming: submission_YYYYMMDD or submission_YYYYMMDD_i
os.makedirs("../data/submission", exist_ok=True)

date_str = datetime.now().strftime("%Y%m%d")
base_name = f"submission_{date_str}"
submission_dir = "../data/submission"

submission_path = os.path.join(submission_dir, f"{base_name}.csv")
i = 1
while os.path.exists(submission_path):
    submission_path = os.path.join(submission_dir, f"{base_name}_{i}.csv")
    i += 1

submission_final.to_csv(submission_path, index=False)

print(f"✅ Submission file saved to {submission_path}")
print(f"\nSubmission shape: {submission_final.shape}")
print(f"\nFirst 10 rows:")
print(submission_final.head(10))

✅ Submission file saved to ../data/submission\submission_20260311_1.csv

Submission shape: (200, 6)

First 10 rows:
    Latitude  Longitude Sample Date  Total Alkalinity  Electrical Conductance  \
0 -32.043333  27.822778  01-09-2014        114.728720              352.850990   
1 -33.329167  26.077500  16-09-2015         72.809528              376.000753   
2 -32.991639  27.640028  07-05-2015         79.887865              616.595792   
3 -34.096389  24.439167  07-02-2012         56.195506              266.076900   
4 -32.000556  28.581667  01-10-2014        124.278987              404.493900   
5 -32.086390  25.575560  19-07-2013         93.124440              468.082567   
6 -32.000556  28.581667  03-09-2014        115.115954              379.611600   
7 -32.991639  27.640028  02-10-2014         81.203543              366.258944   
8 -32.000556  28.581667  06-08-2014        112.658931              297.167517   
9 -33.185361  27.390750  22-09-2011        138.996111              629.597

### Diagnostic

In [20]:
# Check what features the model actually expects vs what you're giving it
print("Expected feature count:", len(preprocessor.get_feature_names_out()))
print("Expected features:", preprocessor.get_feature_names_out()[:20])

print("\nValidation data columns:", df_val_features.columns.tolist())
print("Validation feature count:", len(df_val_features.columns))

Expected feature count: 4
Expected features: ['num__swir22' 'num__NDMI' 'num__MNDWI' 'num__pet']

Validation data columns: ['swir22', 'NDMI', 'MNDWI', 'pet']
Validation feature count: 4


In [21]:
file_a = "../data/submission/submission_20260311.csv"     # <-- set manually
file_b = "../data/submission/submission_20260311_1.csv"   # <-- set manually

df_a = pd.read_csv(file_a)
df_b = pd.read_csv(file_b)

print(f"A shape: {df_a.shape} | {file_a}")
print(f"B shape: {df_b.shape} | {file_b}")

if list(df_a.columns) != list(df_b.columns):
    print("\n⚠️ Columns differ between files.")
    print("Only in A:", sorted(set(df_a.columns) - set(df_b.columns)))
    print("Only in B:", sorted(set(df_b.columns) - set(df_a.columns)))
else:
    print("\n✅ Columns match.")

common_cols = [c for c in df_a.columns if c in df_b.columns]
n = min(len(df_a), len(df_b))
a = df_a.loc[:n-1, common_cols].copy()
b = df_b.loc[:n-1, common_cols].copy()

# Exact equality by column
exact_equal = (a == b) | (a.isna() & b.isna())
equal_rate = exact_equal.mean().sort_values()

print("\n=== Equality rate by column ===")
print((equal_rate * 100).round(2).astype(str) + "%")

# Numeric diff summary
num_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c])]
if num_cols:
    abs_diff = (a[num_cols] - b[num_cols]).abs()
    print("\n=== Numeric absolute difference summary ===")
    print(pd.DataFrame({
        "max_abs_diff": abs_diff.max(),
        "mean_abs_diff": abs_diff.mean(),
        "changed_rows": (abs_diff > 0).sum()
    }).sort_values("max_abs_diff", ascending=False))
else:
    print("\nNo common numeric columns to diff.")

# Optional: row-level mismatch preview
mismatch_mask = ~exact_equal.all(axis=1)
print(f"\nRows with any mismatch: {mismatch_mask.sum()} / {n}")

if mismatch_mask.any():
    preview_cols = common_cols[:8]  # limit display
    preview = pd.concat(
        [a.loc[mismatch_mask, preview_cols].add_prefix("A_"),
         b.loc[mismatch_mask, preview_cols].add_prefix("B_")],
        axis=1
    ).head(10)
    print("\nMismatch preview (first 10 rows):")
    display(preview)

A shape: (200, 6) | ../data/submission/submission_20260311.csv
B shape: (200, 6) | ../data/submission/submission_20260311_1.csv

✅ Columns match.

=== Equality rate by column ===
Total Alkalinity                   0.5%
Electrical Conductance             2.0%
Dissolved Reactive Phosphorus      2.5%
Latitude                         100.0%
Sample Date                      100.0%
Longitude                        100.0%
dtype: object

=== Numeric absolute difference summary ===
                               max_abs_diff  mean_abs_diff  changed_rows
Electrical Conductance            43.427000      10.090505           196
Total Alkalinity                  14.459540       2.225657           199
Dissolved Reactive Phosphorus      9.853333       1.576158           195
Latitude                           0.000000       0.000000             0
Longitude                          0.000000       0.000000             0

Rows with any mismatch: 200 / 200

Mismatch preview (first 10 rows):


,A_Latitude,A_Longitude,A_Sample Date,A_Total Alkalinity,A_Electrical Conductance,A_Dissolved Reactive Phosphorus,B_Latitude,B_Longitude,B_Sample Date,B_Total Alkalinity,B_Electrical Conductance,B_Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,120.385684,354.666183,25.311333,-32.043333,27.822778,01-09-2014,114.728720,352.850990,24.987167
1,-33.329167,26.077500,16-09-2015,74.010598,376.151562,66.065190,-33.329167,26.077500,16-09-2015,72.809528,376.000753,65.905190
2,-32.991639,27.640028,07-05-2015,77.930603,607.061120,26.570167,-32.991639,27.640028,07-05-2015,79.887865,616.595792,27.856167
3,-34.096389,24.439167,07-02-2012,56.195506,265.946600,10.000000,-34.096389,24.439167,07-02-2012,56.195506,266.076900,10.000000
4,-32.000556,28.581667,01-10-2014,125.084762,424.109500,39.480000,-32.000556,28.581667,01-10-2014,124.278987,404.493900,40.260000
5,-32.086390,25.575560,19-07-2013,95.874840,467.043013,35.667714,-32.086390,25.575560,19-07-2013,93.124440,468.082567,33.502548
6,-32.000556,28.581667,03-09-2014,112.571366,399.771700,38.150000,-32.000556,28.581667,03-09-2014,115.115954,379.611600,38.240000
7,-32.991639,27.640028,02-10-2014,83.580578,356.853033,15.977905,-32.991639,27.640028,02-10-2014,81.203543,366.258944,18.959571
8,-32.000556,28.581667,06-08-2014,112.879688,303.224900,19.230000,-32.000556,28.581667,06-08-2014,112.658931,297.167517,18.936667
9,-33.185361,27.390750,22-09-2011,138.996111,629.597016,150.680317,-33.185361,27.390750,22-09-2011,138.996111,629.597016,150.680317
